# 🌤️ Weather Model Training - Dual Model (Hourly + Daily)

Notebook ini mengikuti panduan dari `training_guide.md` untuk melatih **dual-model**:
1. **Model Hourly** - Prediksi per-jam (temp, humidity, windspeed, pressure, weather_code)
2. **Model Daily** - Prediksi per-hari (temp_min, temp_max, temp_mean, humidity_avg, windspeed_avg, pressure_avg, weather_code_dominant)

**Output:** 7 file model `.pkl` untuk berbagai kebutuhan deployment.

## 1. Persiapan Lingkungan dan Pemuatan Pustaka

In [ ]:
# Install dependencies jika belum ada
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost joblib

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os
warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, classification_report, confusion_matrix
)

# Regression Models
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor

# Classification Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# XGBoost
try:
    from xgboost import XGBRegressor, XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    print("XGBoost not installed. Skipping XGBoost models.")
    XGBOOST_AVAILABLE = False

# Joblib for saving models
import joblib

print("✅ Semua pustaka berhasil diimpor!")
print(f"   - Pandas: {pd.__version__}")
print(f"   - NumPy: {np.__version__}")
print(f"   - XGBoost Available: {XGBOOST_AVAILABLE}")

## 2. Pengumpulan dan Pemuatan Data

In [ ]:
# Load dataset (23 kolom: hourly + daily features)
DATA_PATH = '../data/historical_data_2000_2024.csv'

df = pd.read_csv(DATA_PATH)

# Konversi timestamp ke datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Urutkan berdasarkan waktu (PENTING untuk time-series)
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"📊 Dataset loaded: {len(df):,} baris x {len(df.columns)} kolom")
print(f"📅 Rentang waktu: {df['timestamp'].min()} - {df['timestamp'].max()}")
print(f"\n📋 Kolom dataset:")
print(df.columns.tolist())
df.head()

In [ ]:
# Info struktur data
df.info()

## 3. Analisis Data Eksplorasi (EDA)

### 3.1 Statistik Deskriptif

In [ ]:
# Statistik deskriptif untuk fitur numerik
df.describe()

### 3.2 Visualisasi Distribusi

In [ ]:
# Visualisasi distribusi parameter cuaca utama (Hourly)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

params = ['temp', 'humidity', 'windspeed', 'sealevelpressure']
titles = ['Temperature (°C)', 'Humidity (%)', 'Wind Speed (km/h)', 'Sea Level Pressure (hPa)']

for ax, param, title in zip(axes.flatten(), params, titles):
    sns.histplot(df[param], kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribusi {title}')
    ax.set_xlabel(title)

plt.suptitle('Distribusi Parameter Cuaca Hourly', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualisasi distribusi parameter cuaca Daily
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

daily_params = ['temp_max_daily', 'temp_min_daily', 'temp_mean_daily', 
                'humidity_avg_daily', 'pressure_avg_daily', 'windspeed_avg_daily']
titles = ['Temp Max (°C)', 'Temp Min (°C)', 'Temp Mean (°C)', 
          'Humidity Avg (%)', 'Pressure Avg (hPa)', 'Windspeed Avg (km/h)']

for ax, param, title in zip(axes.flatten(), daily_params, titles):
    sns.histplot(df[param].dropna(), kde=True, ax=ax, color='coral')
    ax.set_title(f'Distribusi {title}')
    ax.set_xlabel(title)

plt.suptitle('Distribusi Parameter Cuaca Daily', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.3 Analisis Korelasi

In [ ]:
# Heatmap korelasi
numeric_cols = ['temp', 'humidity', 'windspeed', 'sealevelpressure', 'rain', 
                'weather_code', 'temp_max_daily', 'temp_min_daily', 'temp_mean_daily',
                'humidity_avg_daily', 'pressure_avg_daily', 'windspeed_avg_daily']

plt.figure(figsize=(14, 10))
correlation_matrix = df[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Heatmap Korelasi Antar Variabel Cuaca (Hourly + Daily)')
plt.tight_layout()
plt.show()

### 3.4 Analisis Korelasi: weather_code dan rain

In [ ]:
# Analisis hubungan weather_code dengan rain
weather_rain_analysis = df.groupby('weather_code')[['rain']].agg(['mean', 'min', 'max', 'count'])
print("📊 Weather Code vs Rain:")
weather_rain_analysis

In [ ]:
# Verifikasi korelasi deterministik
print(f"\n🔍 Verifikasi rain == precipitation: {(df['rain'] == df['precipitation']).all()}")
print(f"🔍 Weather codes dengan rain > 0: {sorted(df[df['rain'] > 0]['weather_code'].unique())}")
print(f"🔍 Weather codes dengan rain = 0: {sorted(df[df['rain'] == 0]['weather_code'].unique())}")

# Kesimpulan
print("\n✅ KESIMPULAN:")
print("   - rain dan precipitation IDENTIK di seluruh dataset")
print("   - weather_code >= 50 SELALU hujan (deterministik)")
print("   - Tidak perlu memprediksi rain terpisah, cukup prediksi weather_code")

## 4. Pra-pemrosesan Data dan Feature Engineering

### 4.1 Preprocessing Data Hourly

In [ ]:
# Copy dataframe untuk preprocessing
df_hourly = df.copy()

# 1. Label Encoding untuk 'conditions'
le_conditions = LabelEncoder()
df_hourly['conditions_encoded'] = le_conditions.fit_transform(df_hourly['conditions'])

print("📝 Label Encoding untuk 'conditions':")
for i, label in enumerate(le_conditions.classes_):
    print(f"   {i}: {label}")

In [ ]:
# 2. Label Encoding untuk 'weather_code' (PENTING untuk XGBoost)
# XGBoost membutuhkan label berupa integer berurutan (0, 1, 2, ...)
le_weather_code = LabelEncoder()
df_hourly['weather_code_encoded'] = le_weather_code.fit_transform(df_hourly['weather_code'])

print("📝 Label Encoding untuk 'weather_code':")
for i, label in enumerate(le_weather_code.classes_):
    print(f"   {i}: {label}")

In [ ]:
# 3. Feature Engineering: Lag Features untuk Hourly
hourly_target_cols = ['temp', 'humidity', 'windspeed', 'sealevelpressure']

for col in hourly_target_cols:
    # Lag 1 jam
    df_hourly[f'{col}_lag_1'] = df_hourly[col].shift(1)
    # Lag 24 jam
    df_hourly[f'{col}_lag_24'] = df_hourly[col].shift(24)
    # Rolling mean 24 jam
    df_hourly[f'{col}_rolling_24'] = df_hourly[col].rolling(window=24).mean()

print(f"✅ Feature Engineering Hourly selesai! Kolom baru: {12} fitur lag & rolling")

In [ ]:
# 4. Hapus baris dengan NaN (akibat lag & rolling)
rows_before = len(df_hourly)
df_hourly = df_hourly.dropna().reset_index(drop=True)
rows_after = len(df_hourly)

print(f"🗑️ Baris dihapus (NaN): {rows_before - rows_after:,}")
print(f"📊 Dataset Hourly final: {rows_after:,} baris")

### 4.2 Preprocessing Data Daily

In [ ]:
# Agregasi data hourly menjadi daily
df_daily = df.groupby(['year', 'month', 'day']).agg({
    'temp': ['min', 'max', 'mean'],
    'humidity': 'mean',
    'windspeed': 'mean',
    'sealevelpressure': 'mean',
    'weather_code': lambda x: x.mode()[0],  # Dominan weather_code
    'rain': 'sum'  # Total curah hujan
}).reset_index()

# Flatten column names
df_daily.columns = ['year', 'month', 'day', 
                    'temp_min', 'temp_max', 'temp_mean',
                    'humidity_avg', 'windspeed_avg', 'pressure_avg',
                    'weather_code_dominant', 'rain_total']

print(f"📊 Dataset Daily: {len(df_daily):,} baris (hari)")
df_daily.head()

In [ ]:
# Label Encoding untuk 'weather_code_dominant' (PENTING untuk XGBoost)
le_weather_code_daily = LabelEncoder()
df_daily['weather_code_dominant_encoded'] = le_weather_code_daily.fit_transform(df_daily['weather_code_dominant'])

print("📝 Label Encoding untuk 'weather_code_dominant':")
for i, label in enumerate(le_weather_code_daily.classes_):
    print(f"   {i}: {label}")

In [ ]:
# Feature Engineering Daily - Lag Features
df_daily['temp_min_lag_1'] = df_daily['temp_min'].shift(1)   # Kemarin
df_daily['temp_max_lag_1'] = df_daily['temp_max'].shift(1)
df_daily['temp_mean_lag_1'] = df_daily['temp_mean'].shift(1)
df_daily['humidity_avg_lag_1'] = df_daily['humidity_avg'].shift(1)
df_daily['windspeed_avg_lag_1'] = df_daily['windspeed_avg'].shift(1)
df_daily['pressure_avg_lag_1'] = df_daily['pressure_avg'].shift(1)

df_daily['temp_min_lag_7'] = df_daily['temp_min'].shift(7)   # Seminggu lalu
df_daily['temp_max_lag_7'] = df_daily['temp_max'].shift(7)
df_daily['temp_mean_lag_7'] = df_daily['temp_mean'].shift(7)
df_daily['rain_total_lag_1'] = df_daily['rain_total'].shift(1)

# Hapus NaN
rows_before = len(df_daily)
df_daily = df_daily.dropna().reset_index(drop=True)
rows_after = len(df_daily)

print(f"🗑️ Baris dihapus (NaN): {rows_before - rows_after:,}")
print(f"📊 Dataset Daily final: {rows_after:,} baris")

## 5. Pelatihan dan Perbandingan Model

### 5.1 Pemisahan Data (Time-Series Split)

In [ ]:
# ===== HOURLY DATA SPLIT =====
hourly_train_size = int(len(df_hourly) * 0.8)
hourly_train = df_hourly[:hourly_train_size]
hourly_test = df_hourly[hourly_train_size:]

print(f"📊 HOURLY Data Split (80-20):")
print(f"   Train: {len(hourly_train):,} baris")
print(f"   Test:  {len(hourly_test):,} baris")

# ===== DAILY DATA SPLIT =====
daily_train_size = int(len(df_daily) * 0.8)
daily_train = df_daily[:daily_train_size]
daily_test = df_daily[daily_train_size:]

print(f"\n📊 DAILY Data Split (80-20):")
print(f"   Train: {len(daily_train):,} baris")
print(f"   Test:  {len(daily_test):,} baris")

In [ ]:
# ===== DEFINISI FITUR DAN TARGET =====

# HOURLY Features
hourly_feature_cols = [
    'year', 'month', 'day', 'hour',
    'temp_lag_1', 'temp_lag_24', 'temp_rolling_24',
    'humidity_lag_1', 'humidity_lag_24', 'humidity_rolling_24',
    'windspeed_lag_1', 'windspeed_lag_24', 'windspeed_rolling_24',
    'sealevelpressure_lag_1', 'sealevelpressure_lag_24', 'sealevelpressure_rolling_24'
]
hourly_target_reg = ['temp', 'humidity', 'windspeed', 'sealevelpressure']
hourly_target_clf = 'weather_code_encoded'  # Gunakan encoded untuk XGBoost!

# DAILY Features
daily_feature_cols = [
    'year', 'month', 'day',
    'temp_min_lag_1', 'temp_max_lag_1', 'temp_mean_lag_1',
    'humidity_avg_lag_1', 'windspeed_avg_lag_1', 'pressure_avg_lag_1',
    'temp_min_lag_7', 'temp_max_lag_7', 'temp_mean_lag_7',
    'rain_total_lag_1'
]
daily_target_reg = ['temp_min', 'temp_max', 'temp_mean', 'humidity_avg', 'windspeed_avg', 'pressure_avg']
daily_target_clf = 'weather_code_dominant_encoded'  # Gunakan encoded untuk XGBoost!

# Pisahkan X dan y untuk HOURLY
X_hourly_train = hourly_train[hourly_feature_cols]
X_hourly_test = hourly_test[hourly_feature_cols]
y_hourly_train_reg = hourly_train[hourly_target_reg]
y_hourly_test_reg = hourly_test[hourly_target_reg]
y_hourly_train_clf = hourly_train[hourly_target_clf]
y_hourly_test_clf = hourly_test[hourly_target_clf]

# Pisahkan X dan y untuk DAILY
X_daily_train = daily_train[daily_feature_cols]
X_daily_test = daily_test[daily_feature_cols]
y_daily_train_reg = daily_train[daily_target_reg]
y_daily_test_reg = daily_test[daily_target_reg]
y_daily_train_clf = daily_train[daily_target_clf]
y_daily_test_clf = daily_test[daily_target_clf]

print(f"✅ HOURLY Features: {len(hourly_feature_cols)} | Targets Reg: {hourly_target_reg} | Target Clf: {hourly_target_clf}")
print(f"✅ DAILY Features: {len(daily_feature_cols)} | Targets Reg: {daily_target_reg} | Target Clf: {daily_target_clf}")

### 5.2 Komparasi Model Regresi (Hourly)

In [ ]:
# # Definisi model regresi
# regression_models = {
#     'Linear Regression': LinearRegression(),
#     'K-Neighbors': KNeighborsRegressor(n_neighbors=5),
#     'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=10),
#     'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
# }

# if XGBOOST_AVAILABLE:
#     regression_models['XGBoost'] = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbosity=0)


In [ ]:
regression_models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "K-Neighbors": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=5))
    ]),
    # Tree-based biasanya tidak butuh scaling
    "Decision Tree": DecisionTreeRegressor(
        random_state=42,
        max_depth=10
    ),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),
}

if XGBOOST_AVAILABLE:

    # XGBoost dibungkus MultiOutputRegressor supaya bisa multi-target
    regression_models["XGBoost"] = MultiOutputRegressor(
        XGBRegressor(
            n_estimators=100,
            random_state=42,
            n_jobs=-1,
            verbosity=0,
            tree_method="hist"
        )
    )

regression_models

In [ ]:
# Fungsi evaluasi
def evaluate_regression(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'R2': r2}

In [ ]:
# Training HOURLY Regression Models
print("="*70)
print("🔄 TRAINING HOURLY REGRESSION MODELS")
print("="*70)

hourly_reg_results = []

for name, model in regression_models.items():
    print(f"\n🔄 Training {name}...")
    model.fit(X_hourly_train, y_hourly_train_reg)
    y_pred = model.predict(X_hourly_test)
    
    overall_metrics = evaluate_regression(y_hourly_test_reg, y_pred)
    overall_metrics['Model'] = name
    hourly_reg_results.append(overall_metrics)
    
    print(f"   ✅ {name} - R²: {overall_metrics['R2']:.4f}, RMSE: {overall_metrics['RMSE']:.4f}")

df_hourly_reg = pd.DataFrame(hourly_reg_results).sort_values('R2', ascending=False)
print("\n📊 HASIL PERBANDINGAN REGRESI HOURLY:")
display(df_hourly_reg[['Model', 'R2', 'RMSE', 'MAE']].reset_index(drop=True))

best_hourly_reg_model = df_hourly_reg.iloc[0]['Model']
print(f"\n🏆 MODEL TERBAIK REGRESI HOURLY: {best_hourly_reg_model}")

### 5.3 Komparasi Model Klasifikasi (Hourly)

In [ ]:
# # Definisi model klasifikasi
# classification_models = {
#     'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
#     'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10),
#     'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
# }

# if XGBOOST_AVAILABLE:
#     # XGBoost membutuhkan num_class untuk multiclass klasifikasi
#     num_classes_hourly = len(le_weather_code.classes_)
#     classification_models['XGBoost'] = XGBClassifier(
#         n_estimators=100, 
#         random_state=42, 
#         n_jobs=-1, 
#         verbosity=0,
#         objective='multi:softmax',
#         num_class=num_classes_hourly
#     )

In [ ]:
if XGBOOST_AVAILABLE:
    # XGBoost membutuhkan num_class untuk multiclass klasifikasi
    num_classes_hourly = len(le_weather_code.classes_)

In [ ]:
classification_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            multi_class="multinomial",
            n_jobs=-1
        ))
    ]),
    "Decision Tree": DecisionTreeClassifier(
        random_state=42,
        max_depth=None
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
}

if XGBOOST_AVAILABLE:

    classification_models["XGBoost"] = XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="multi:softprob",
        num_class=num_classes_hourly,  # sudah kamu definisikan sebelumnya
        tree_method="hist",
        n_jobs=-1,
        eval_metric="mlogloss"
    )

classification_models

In [ ]:
# Training HOURLY Classification Models
print("="*70)
print("🔄 TRAINING HOURLY CLASSIFICATION MODELS")
print("="*70)

hourly_clf_results = []

for name, model in classification_models.items():
    print(f"\n🔄 Training {name}...")
    model.fit(X_hourly_train, y_hourly_train_clf)
    y_pred = model.predict(X_hourly_test)
    
    accuracy = accuracy_score(y_hourly_test_clf, y_pred)
    f1_weighted = f1_score(y_hourly_test_clf, y_pred, average='weighted', zero_division=0)
    
    hourly_clf_results.append({
        'Model': name,
        'Accuracy': accuracy,
        'F1 (Weighted)': f1_weighted
    })
    
    print(f"   ✅ {name} - Accuracy: {accuracy:.4f}, F1: {f1_weighted:.4f}")

df_hourly_clf = pd.DataFrame(hourly_clf_results).sort_values('Accuracy', ascending=False)
print("\n📊 HASIL PERBANDINGAN KLASIFIKASI HOURLY:")
display(df_hourly_clf.reset_index(drop=True))

best_hourly_clf_model = df_hourly_clf.iloc[0]['Model']
print(f"\n🏆 MODEL TERBAIK KLASIFIKASI HOURLY: {best_hourly_clf_model}")

### 5.4 Komparasi Model Regresi (Daily)

In [ ]:
# Training DAILY Regression Models
print("="*70)
print("🔄 TRAINING DAILY REGRESSION MODELS")
print("="*70)

daily_reg_results = []

# Re-initialize models
regression_models_daily = {
    'Linear Regression': LinearRegression(),
    'K-Neighbors': KNeighborsRegressor(n_neighbors=5),
    'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=10),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}
if XGBOOST_AVAILABLE:
    regression_models_daily['XGBoost'] = XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbosity=0)

for name, model in regression_models_daily.items():
    print(f"\n🔄 Training {name}...")
    model.fit(X_daily_train, y_daily_train_reg)
    y_pred = model.predict(X_daily_test)
    
    overall_metrics = evaluate_regression(y_daily_test_reg, y_pred)
    overall_metrics['Model'] = name
    daily_reg_results.append(overall_metrics)
    
    print(f"   ✅ {name} - R²: {overall_metrics['R2']:.4f}, RMSE: {overall_metrics['RMSE']:.4f}")

df_daily_reg = pd.DataFrame(daily_reg_results).sort_values('R2', ascending=False)
print("\n📊 HASIL PERBANDINGAN REGRESI DAILY:")
display(df_daily_reg[['Model', 'R2', 'RMSE', 'MAE']].reset_index(drop=True))

best_daily_reg_model = df_daily_reg.iloc[0]['Model']
print(f"\n🏆 MODEL TERBAIK REGRESI DAILY: {best_daily_reg_model}")

### 5.5 Komparasi Model Klasifikasi (Daily)

In [ ]:
# Training DAILY Classification Models
print("="*70)
print("🔄 TRAINING DAILY CLASSIFICATION MODELS")
print("="*70)

daily_clf_results = []

# Re-initialize models
num_classes_daily = len(le_weather_code_daily.classes_)
classification_models_daily = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
}
if XGBOOST_AVAILABLE:
    classification_models_daily['XGBoost'] = XGBClassifier(
        n_estimators=100, 
        random_state=42, 
        n_jobs=-1, 
        verbosity=0,
        objective='multi:softmax',
        num_class=num_classes_daily
    )

for name, model in classification_models_daily.items():
    print(f"\n🔄 Training {name}...")
    model.fit(X_daily_train, y_daily_train_clf)
    y_pred = model.predict(X_daily_test)
    
    accuracy = accuracy_score(y_daily_test_clf, y_pred)
    f1_weighted = f1_score(y_daily_test_clf, y_pred, average='weighted', zero_division=0)
    
    daily_clf_results.append({
        'Model': name,
        'Accuracy': accuracy,
        'F1 (Weighted)': f1_weighted
    })
    
    print(f"   ✅ {name} - Accuracy: {accuracy:.4f}, F1: {f1_weighted:.4f}")

df_daily_clf = pd.DataFrame(daily_clf_results).sort_values('Accuracy', ascending=False)
print("\n📊 HASIL PERBANDINGAN KLASIFIKASI DAILY:")
display(df_daily_clf.reset_index(drop=True))

best_daily_clf_model = df_daily_clf.iloc[0]['Model']
print(f"\n🏆 MODEL TERBAIK KLASIFIKASI DAILY: {best_daily_clf_model}")

### 5.9 Kesimpulan Pemilihan Model

In [ ]:
print("="*70)
print("🎯 KESIMPULAN PEMILIHAN MODEL TERBAIK")
print("="*70)

print("\n📈 MODEL HOURLY:")
print(f"   🏆 Regresi: {best_hourly_reg_model} (R²: {df_hourly_reg.iloc[0]['R2']:.4f})")
print(f"   🏆 Klasifikasi: {best_hourly_clf_model} (Acc: {df_hourly_clf.iloc[0]['Accuracy']:.4f})")

print("\n📊 MODEL DAILY:")
print(f"   🏆 Regresi: {best_daily_reg_model} (R²: {df_daily_reg.iloc[0]['R2']:.4f})")
print(f"   🏆 Klasifikasi: {best_daily_clf_model} (Acc: {df_daily_clf.iloc[0]['Accuracy']:.4f})")

print("\n" + "="*70)
print("✅ Langkah selanjutnya: Retraining dengan 100% data, lalu simpan 7 file model")
print("="*70)

## 6.1 Evaluasi Model Regresi Per-Target

In [ ]:
# ====== Fungsi evaluasi regresi yang lebih detail (opsional) ======
def evaluate_regression_detailed(y_true, y_pred, target_names):
    """
    Mengembalikan:
      - overall: hasil evaluate_regression(y_true, y_pred)
      - per_target: dict {nama_target: {RMSE, MAE, R2}}
    """
    overall = evaluate_regression(y_true, y_pred)

    per_target = {}
    for i, col in enumerate(target_names):
        per_target[col] = {
            "RMSE": np.sqrt(mean_squared_error(y_true.iloc[:, i], y_pred[:, i])),
            "MAE": mean_absolute_error(y_true.iloc[:, i], y_pred[:, i]),
            "R2": r2_score(y_true.iloc[:, i], y_pred[:, i]),
        }
    return overall, per_target


In [ ]:
# Get the actual trained model object (not just the name string)
best_model = regression_models[best_hourly_reg_model]
y_pred_hourly_per_target = best_model.predict(X_hourly_test)

overall, per_target = evaluate_regression_detailed(
    y_hourly_test_reg,
    y_pred_hourly_per_target,
    hourly_target_reg
)
overall, per_target

## 7. Evaluasi Prediksi & Akurasi Tahun 2024 (Best Model)

Evaluasi fokus pada subset tahun 2022 menggunakan model terbaik yang dipilih di atas.
- Regresi: MSE, RMSE, MAE, R² (overall dan per target) + visualisasi aktual vs prediksi.
- Klasifikasi: Accuracy, F1 (weighted), laporan per kelas + confusion matrix ternormalisasi.


In [ ]:
# Subset: 1 bulan (Januari) awal tahun 2024
df_hourly_2022 = df_hourly[(df_hourly['year'] == 2024) & (df_hourly['month'] == 1)].copy()
df_daily_2022 = df_daily[(df_daily['year'] == 2024) & (df_daily['month'] == 1)].copy()

print('Subset: Januari 2024:')
print(f"- Hourly: {len(df_hourly_2022):,} baris")
if len(df_hourly_2022):
    print(f"  Range waktu (hourly): {df_hourly_2022['timestamp'].min()} - {df_hourly_2022['timestamp'].max()}")

print(f"- Daily : {len(df_daily_2022):,} baris")
if len(df_daily_2022):
    date_axis_daily = pd.to_datetime(df_daily_2022[['year', 'month', 'day']])
    print(f"  Range waktu (daily): {date_axis_daily.min().date()} - {date_axis_daily.max().date()}")


### 7.1 Hourly Regression (Best Model)

In [ ]:
# Evaluasi regresi hourly pada 2022 (best model)
hourly_reg_best = regression_models[best_hourly_reg_model]
X_hourly_2022 = df_hourly_2022[hourly_feature_cols]
y_hourly_2022_reg = df_hourly_2022[hourly_target_reg]

y_hourly_pred_2022 = hourly_reg_best.predict(X_hourly_2022)

hourly_reg_overall_2022, hourly_reg_per_target_2022 = evaluate_regression_detailed(
    y_hourly_2022_reg,
    y_hourly_pred_2022,
    hourly_target_reg
)

print(f"Model terbaik: {best_hourly_reg_model}")
display(pd.DataFrame([hourly_reg_overall_2022]))

pd.DataFrame(hourly_reg_per_target_2022).T


In [ ]:
# Grafik per-target: aktual vs prediksi (hourly 2022)
fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharex=True)
time_axis = df_hourly_2022['timestamp']

for idx, target in enumerate(hourly_target_reg):
    ax = axes[idx // 2][idx % 2]
    ax.plot(time_axis, y_hourly_2022_reg[target], label='Aktual', alpha=0.7)
    ax.plot(time_axis, y_hourly_pred_2022[:, idx], label='Prediksi', alpha=0.7)
    ax.set_title(f"Hourly 2022 - {target}")
    ax.set_ylabel(target)
    ax.grid(True, alpha=0.3)

axes[0, 0].legend(loc='upper right')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


### 7.2 Daily Regression (Best Model)

In [ ]:
# Evaluasi regresi daily pada 2022 (best model)
daily_reg_best = regression_models_daily[best_daily_reg_model]
X_daily_2022 = df_daily_2022[daily_feature_cols]
y_daily_2022_reg = df_daily_2022[daily_target_reg]

y_daily_pred_2022 = daily_reg_best.predict(X_daily_2022)

daily_reg_overall_2022, daily_reg_per_target_2022 = evaluate_regression_detailed(
    y_daily_2022_reg,
    y_daily_pred_2022,
    daily_target_reg
)

print(f"Model terbaik: {best_daily_reg_model}")
display(pd.DataFrame([daily_reg_overall_2022]))

pd.DataFrame(daily_reg_per_target_2022).T


In [ ]:
# Grafik per-target: aktual vs prediksi (daily 2022)
fig, axes = plt.subplots(2, 3, figsize=(18, 10), sharex=True)
date_axis = pd.to_datetime(df_daily_2022[['year', 'month', 'day']])

for idx, target in enumerate(daily_target_reg):
    ax = axes[idx // 3][idx % 3]
    ax.plot(date_axis, y_daily_2022_reg[target], label='Aktual', alpha=0.7)
    ax.plot(date_axis, y_daily_pred_2022[:, idx], label='Prediksi', alpha=0.7)
    ax.set_title(f"Daily 2022 - {target}")
    ax.set_ylabel(target)
    ax.grid(True, alpha=0.3)

axes[0, 0].legend(loc='upper right')
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


### 7.3 Hourly Classification (Best Model)

In [ ]:
# Evaluasi klasifikasi hourly pada 2022 (best model)
hourly_clf_best = classification_models[best_hourly_clf_model]
y_hourly_2022_clf = df_hourly_2022[hourly_target_clf]

y_hourly_pred_clf_2022 = hourly_clf_best.predict(X_hourly_2022)

hourly_acc_2022 = accuracy_score(y_hourly_2022_clf, y_hourly_pred_clf_2022)
hourly_f1_2022 = f1_score(y_hourly_2022_clf, y_hourly_pred_clf_2022, average='weighted', zero_division=0)

print(f"Model terbaik: {best_hourly_clf_model}")
print(f"Accuracy 2022 : {hourly_acc_2022:.4f}")
print(f"F1 (weighted): {hourly_f1_2022:.4f}")

hourly_report_2022 = classification_report(
    y_hourly_2022_clf,
    y_hourly_pred_clf_2022,
    labels=np.arange(len(le_weather_code.classes_)),
    target_names=le_weather_code.classes_,
    output_dict=True,
    zero_division=0
)
pd.DataFrame(hourly_report_2022).T


In [ ]:
# Confusion matrix ternormalisasi (hourly 2022)
cm_hourly_2022 = confusion_matrix(
    y_hourly_2022_clf,
    y_hourly_pred_clf_2022,
    labels=np.arange(len(le_weather_code.classes_))
)
cm_hourly_norm = cm_hourly_2022.astype(float) / (cm_hourly_2022.sum(axis=1, keepdims=True) + 1e-9)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_hourly_norm,
    cmap='Blues',
    xticklabels=le_weather_code.classes_,
    yticklabels=le_weather_code.classes_,
    fmt='.2f'
)
plt.title(f"Confusion Matrix (Normalized) - Hourly 2022 - {best_hourly_clf_model}")
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.tight_layout()
plt.show()


### 7.4 Daily Classification (Best Model)

In [ ]:
# Evaluasi klasifikasi daily pada 2022 (best model)
daily_clf_best = classification_models_daily[best_daily_clf_model]
y_daily_2022_clf = df_daily_2022[daily_target_clf]

y_daily_pred_clf_2022 = daily_clf_best.predict(X_daily_2022)

daily_acc_2022 = accuracy_score(y_daily_2022_clf, y_daily_pred_clf_2022)
daily_f1_2022 = f1_score(y_daily_2022_clf, y_daily_pred_clf_2022, average='weighted', zero_division=0)

print(f"Model terbaik: {best_daily_clf_model}")
print(f"Accuracy 2022 : {daily_acc_2022:.4f}")
print(f"F1 (weighted): {daily_f1_2022:.4f}")

daily_report_2022 = classification_report(
    y_daily_2022_clf,
    y_daily_pred_clf_2022,
    labels=np.arange(len(le_weather_code_daily.classes_)),
    target_names=le_weather_code_daily.classes_,
    output_dict=True,
    zero_division=0
)
pd.DataFrame(daily_report_2022).T


In [ ]:
# Confusion matrix ternormalisasi (daily 2022)
cm_daily_2022 = confusion_matrix(
    y_daily_2022_clf,
    y_daily_pred_clf_2022,
    labels=np.arange(len(le_weather_code_daily.classes_))
)
cm_daily_norm = cm_daily_2022.astype(float) / (cm_daily_2022.sum(axis=1, keepdims=True) + 1e-9)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_daily_norm,
    cmap='Purples',
    xticklabels=le_weather_code_daily.classes_,
    yticklabels=le_weather_code_daily.classes_,
    fmt='.2f'
)
plt.title(f"Confusion Matrix (Normalized) - Daily 2022 - {best_daily_clf_model}")
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.tight_layout()
plt.show()


## Rangkuman

Notebook ini telah menyelesaikan:

1. ✅ **Persiapan Lingkungan** - Import semua pustaka
2. ✅ **Pemuatan Data** - Load dataset 23 kolom (hourly + daily features)
3. ✅ **EDA** - Analisis distribusi, korelasi, hubungan weather_code dengan rain
4. ✅ **Feature Engineering** - Lag features untuk Hourly dan Daily
5. ✅ **Perbandingan Model** - 5 model regresi & 4 model klasifikasi untuk Hourly dan Daily
6. ✅ **Evaluasi 2022** - Perbandingan best-model pada tahun 2022 (metrik lengkap + grafik per target)

**PENTING:** Label `weather_code` dan `weather_code_dominant` telah di-encode menggunakan `LabelEncoder` untuk kompatibilitas dengan XGBoost. Gunakan `label_encoder.inverse_transform()` untuk mendapatkan nilai asli saat inferensi.
